In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("cuda available =", torch.cuda.is_available())
print("device count =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0 =", torch.cuda.get_device_name(0))

CUDA_VISIBLE_DEVICES = 1
LD_LIBRARY_PATH = None
cuda available = True
device count = 1
device 0 = NVIDIA RTX A6000


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm import HyperbolicLCM


# CONFIG

@dataclass
class GRPOConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_cached_reflogits_fixed"
    cache_dir: str = "arc_cached_features"
    ref_logits_dir: str = "arc_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks" 
    n_last_blocks: int = 1

    train_batch_size: int = 1
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT
    sft_epochs: int = 2
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO
    grpo_epochs: int = 1
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # choice chunking
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "head_only":
        raise ValueError(
            "finetune_mode='head_only' is invalid for this script because no separate trainable head exists. "
            "Use 'last_blocks' or 'full'."
        )
    elif mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")

# CONCEPTIZER

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad

# ARC

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: GRPOConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)
    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


def cache_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }

# MODEL BUILD

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    for k0 in range(0, K, max(1, choice_chunk_size)):
        k1 = min(K, k0 + max(1, choice_chunk_size))

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows

#  SFT

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }

# GRPO

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }

def train_arc_hybrid_hlcm(dataset_name: str, cfg: GRPOConfig, device: torch.device):
    print(f"\n {dataset_name}")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_arc(cfg, dataset_name)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer)
    test_rows = build_or_load_cached_split(cfg, dataset_name, "test", test_hf, conceptizer) if test_hf is not None else []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    # model
    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metadata_base = {
        "dataset": dataset_name,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "num_test_examples": len(test_rows),
    }

    # SFT
    print(f"\n{dataset_name} :: SFT ")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    # GRPO 
    print(f"\n{dataset_name} :: GRPO")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    final_test = evaluate_hlcm(hlcm, test_loader, mu, sigma, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "final_test_loss": None if final_test is None else final_test["loss"],
            "final_test_acc": None if final_test is None else final_test["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "ref_logits_path": ref_logits_path,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] dataset={dataset_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()

# MAIN

def main():
    cfg = GRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_arc_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=1, bs_eval=2, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
choice_chunk_size=1

==================== ARC-Easy ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building ARC-Easy / train


cache:ARC-Easy:train: 100%|█████████████████████████████████████| 2251/2251 [04:07<00:00,  9.08it/s]


...

[SFT][epoch 2/2] train_loss=1.3776 train_acc=0.2817 eval_loss=1.3560 eval_acc=0.3737
  saved sft_best.pt
[SFT][FINAL] best_acc=0.3737 total_train_time=68.13 min
[ref_logits] building arc_cached_ref_logits/ARC-Easy_train_ref_logits_tok256_seq8.pt


precompute_ref_logits: 100%|████████████████████████████████████| 1126/1126 [04:52<00:00,  3.85it/s]


[ref_logits] saved arc_cached_ref_logits/ARC-Easy_train_ref_logits_tok256_seq8.pt (2251 rows)

========== ARC-Easy :: STAGE 2 / GRPO ==========
[GRPO][BASE] loss=1.3560 acc=0.3737


grpo epoch 1/1: 100%|█| 2251/2251 [31:47<00:00,  1.18it/s, acc=0.3034, kl=0.0113, loss=-0.0076, lr=0


[GRPO][epoch 1/1] train_loss=-0.0076 train_acc=0.3034 eval_loss=1.3490 eval_acc=0.3789
  saved grpo_best.pt
[GRPO][FINAL] best_acc=0.3789 total_train_time=33.66 min
[HYBRID][FINAL] dataset=ARC-Easy sft_best_acc=0.3737 grpo_best_acc=0.3789 final_eval_acc=0.3789 final_test_acc=0.3329 max_gpu_alloc=38605.1 MB saved -> runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy

All done. Outputs in: runs/hlcm_arc_cached_reflogits_fixed
Total wall time: 128.81 min


In [1]:
#precision test

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_cached_reflogits_fixed"
    cache_dir: str = "arc_cached_features"

    # pretrained HLCM architecture checkpoint, used only if grpo_best.pt stores partial state
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM architecture
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # eval
    eval_batch_size: int = 2
    num_workers: int = 0

    # logits
    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# ARC DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        # ARC sometimes uses numeric answer keys while labels can be A/B/C/D
        alpha_to_num = {"A": "1", "B": "2", "C": "3", "D": "4", "E": "5"}
        num_to_alpha = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}

        alt = alpha_to_num.get(key, num_to_alpha.get(key, None))
        if alt is not None and alt in labels:
            return labels.index(alt)

    return 0


def load_arc(cfg: EvalConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHED FEATURES
# ============================================================

def cache_file_path(cfg: EvalConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: EvalConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Base HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] base HLCM from {cfg.ckpt_path}")
    print(f"[load] base missing keys: {len(missing)}")
    print(f"[load] base unexpected keys: {len(unexpected)}")

    return model


def load_grpo_checkpoint(model: HyperbolicLCM, grpo_path: str, device: torch.device):
    if not os.path.exists(grpo_path):
        raise FileNotFoundError(f"grpo_best.pt not found: {grpo_path}")

    obj = torch.load(grpo_path, map_location="cpu")

    if isinstance(obj, dict) and "model" in obj:
        state = obj["model"]
    elif isinstance(obj, dict) and "model_state" in obj:
        state = obj["model_state"]
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] GRPO checkpoint from {grpo_path}")
    print(f"[load] GRPO missing keys: {len(missing)}")
    print(f"[load] GRPO unexpected keys: {len(unexpected)}")

    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    info = {}

    if isinstance(obj, dict):
        for key in ["stage", "epoch", "best_eval_acc", "global_opt_step", "dataset"]:
            if key in obj:
                info[key] = obj[key]

    return info


# ============================================================
# LOGITS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk = max(1, int(choice_chunk_size))

    for k0 in range(0, K, chunk):
        k1 = min(K, k0 + chunk)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits

def multiclass_brier_score(probs: torch.Tensor, labels: torch.Tensor, choice_mask: torch.Tensor) -> torch.Tensor:
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq.sum(dim=1)


def expected_calibration_error(confidences: torch.Tensor, correctness: torch.Tensor, n_bins: int = 15):
    ece = 0.0
    mce = 0.0

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)

# ============================================================
# RANKING METRICS
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics_hlcm(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4, 5),
) -> Dict[str, float]:
    model.eval()

    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}

    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        batch_size = labels.size(0)
        total += batch_size
        total_loss += float(loss.item()) * batch_size

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        total_brier += float(
            multiclass_brier_score(probs, labels, choice_mask).sum().item()
        )

        all_confidences.append(probs.max(dim=-1).values.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0

                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences = torch.cat(all_confidences) if all_confidences else torch.empty(0)
    correctness = torch.cat(all_correctness) if all_correctness else torch.empty(0)

    ece, mce = expected_calibration_error(
        confidences=confidences,
        correctness=correctness,
        n_bins=15,
    )

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results

# ============================================================
# EVAL ONE DATASET
# ============================================================

def eval_one_dataset(dataset_name: str, cfg: EvalConfig, device: torch.device):
    print(f"\n==================== EVAL ONLY: {dataset_name} ====================")

    dataset_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    grpo_path = os.path.join(dataset_dir, "grpo_best.pt")

    eval_hf, test_hf = load_arc(cfg, dataset_name)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    test_rows = []
    if test_hf is not None:
        test_rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=dataset_name,
            split_name="test",
            hf_split=test_hf,
            conceptizer=conceptizer,
        )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_loader = DataLoader(
            CachedMCQDataset(test_rows),
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    model = load_base_hlcm(cfg, device)
    ckpt_info = load_grpo_checkpoint(model, grpo_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    validation_metrics = evaluate_ranking_metrics_hlcm(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4, 5),
    )

    test_metrics = None
    if test_loader is not None:
        test_metrics = evaluate_ranking_metrics_hlcm(
            model=model,
            loader=test_loader,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            k_values=(1, 2, 3, 4, 5),
        )

    summary = {
        "dataset": dataset_name,
        "checkpoint": grpo_path,
        "checkpoint_info": ckpt_info,
        "num_validation_examples": len(eval_rows),
        "num_test_examples": len(test_rows),
        "validation": validation_metrics,
        "test": test_metrics,
    }

    ensure_dir(dataset_dir)

    save_path = os.path.join(dataset_dir, "grpo_eval_precision_recall_ranking_only.json")
    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    del model
    cuda_cleanup()

    return summary


# ============================================================
# RUN EVAL ONLY
# ============================================================

def main():
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("This script is EVAL ONLY. It will not train.")

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    all_results = {}

    for dataset_name in cfg.datasets_to_run:
        all_results[dataset_name] = eval_one_dataset(dataset_name, cfg, device)

    save_path = os.path.join(cfg.out_dir, "grpo_eval_precision_recall_ranking_only_all.json")

    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_results, indent=2))
    print(f"[saved] {save_path}")


if __name__ == "__main__":
    main()

Device: cuda:0
This script is EVAL ONLY. It will not train.

==================== EVAL ONLY: ARC-Easy ====================


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Easy' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Easy/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:57:08 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading arc_cached_features/ARC-Easy_validation_tok256_seq8.pt
[cache] loading arc_cached_features/ARC-Easy_test_tok256_seq8.pt
[load] base HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] base missing keys: 0
[load] base unexpected keys: 0
[load] GRPO checkpoint from runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt
[load] GRPO missing keys: 0
[load] GRPO unexpected keys: 0


{
  "dataset": "ARC-Easy",
  "checkpoint": "runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt",
  "checkpoint_info": {
    "stage": "grpo_best",
    "epoch": 1,
    "best_eval_acc": 0.37894736842105264,
    "global_opt_step": 282,
    "dataset": "ARC-Easy"
  },
  "num_validation_examples": 570,
  "num_test_examples": 2376,
  "validation": {
    "loss": 1.3489915684649818,
    "accuracy": 0.37894736842105264,
    "brier_score": 0.7305204180248996,
    "ece": 0.08640435933133653,
    "mce": 0.29005634784698486,
    "ece_bins": 15,
    "precision@1": 0.37894736842105264,
    "recall@1": 0.37894736842105264,
    "precision@2": 0.3078947368421053,
    "recall@2": 0.6157894736842106,
    "precision@3": 0.27368421052631586,
    "recall@3": 0.8210526315789474,
    "precision@4": 0.25014619883040934,
    "recall@4": 1.0,
    "precision@5": 0.24997076023391815,
    "recall@5": 1.0,
    "mrr": 0.6105263157894731
  },
  "test": {
    "loss": 1.3568283150894473,
    "accuracy": 0.332912457

In [1]:
#Inference 
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "ARC-Easy"
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_cached_reflogits_fixed"
    cache_dir: str = "arc_cached_features"

    # This is your GRPO checkpoint
    grpo_ckpt_path: str = "runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt"

    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0

    # use "validation" or "test"
    split: str = "test"

    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# ARC DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: InferenceConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})

    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])

    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHE FEATURES
# ============================================================

def cache_file_path(cfg: InferenceConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(f"Cache not found: {path}")

    if conceptizer is None:
        raise RuntimeError("Conceptizer required because cache is missing.")

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": choices,
                    "cmask": cmask,
                    "choice_mask": choice_mask,
                    "label": int(label),
                    "num_choices": int(K),
                }
            )

        except Exception as e:
            skipped += 1
            print(
                f"[warn] skipped one example in {dataset_name}-{split_name}: "
                f"{type(e).__name__}: {e}"
            )

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]

        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_hlcm(cfg: InferenceConfig, device: torch.device) -> HyperbolicLCM:
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(f"GRPO checkpoint not found: {cfg.grpo_ckpt_path}")

    model = build_hlcm_from_cfg(cfg).to(device)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_best.pt does not contain key 'model'.")

    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded GRPO model from {cfg.grpo_ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape

    out = torch.empty(
        (B, D),
        device=h_tan.device,
        dtype=h_tan.dtype,
    )

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(
        model=model,
        x=q,
        pad_mask=qmask,
        mu=mu,
        sigma=sigma,
    )
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []

    for k0 in range(0, K, max(1, cfg.choice_chunk_size)):
        k1 = min(K, k0 + max(1, cfg.choice_chunk_size))

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(
            model=model,
            x=flat,
            pad_mask=flat_mask,
            mu=mu,
            sigma=sigma,
        )

        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits


# ============================================================
# INFERENCE + TIME
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
    save_path: Optional[str] = None,
) -> Dict[str, Any]:
    model.eval()

    device = next(model.parameters()).device

    total = 0
    correct = 0
    predictions = []
    total_loss = 0.0

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference {cfg.dataset_name}-{cfg.split}"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)

        total_loss += float(loss.item()) * bs
        correct += int((preds == labels).sum().item())
        total += int(bs)

        for i in range(bs):
            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(preds[i].item() == labels[i].item()),
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "dataset": cfg.dataset_name,
        "split": cfg.split,
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_arc(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[dataset]", cfg.dataset_name)
    print("[split]", cfg.split)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    train_hf, eval_hf, test_hf = load_arc(cfg, cfg.dataset_name)

    if cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    elif cfg.split in {"validation", "val", "dev"}:
        hf_split = eval_hf
        split_name = "validation"
    elif cfg.split == "test":
        if test_hf is None:
            raise RuntimeError("No test split available.")
        hf_split = test_hf
        split_name = "test"
    else:
        raise ValueError("cfg.split must be one of: train, validation, test")

    cache_path = cache_file_path(cfg, cfg.dataset_name, split_name)

    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer_device = torch.device(cfg.conceptizer_device)
        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=conceptizer_device,
        )

        cache_t0 = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=conceptizer,
        )

        cache_build_time_sec = time.perf_counter() - cache_t0

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=None,
        )

    print(f"[data] examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model = load_grpo_hlcm(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    save_path = os.path.join(
        dataset_out_dir,
        f"grpo_inference_{split_name}_results.json",
    )

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "dataset": cfg.dataset_name,
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"grpo_inference_{split_name}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== GRPO INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [2]:
cfg = InferenceConfig(
    dataset_name="ARC-Easy",

    grpo_ckpt_path="runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt",
    normalizer_path="normalizer.pt",

    out_dir="runs/hlcm_arc_cached_reflogits_fixed",
    cache_dir="arc_cached_features",

    split="test",
    eval_batch_size=2,
    choice_chunk_size=1,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_arc(cfg)

[device] cuda:0
[dataset] ARC-Easy
[split] test


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Easy' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Easy/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:57:08 2026).


[cache] loading arc_cached_features/ARC-Easy_test_tok256_seq8.pt
[data] examples=2376
[cache] arc_cached_features/ARC-Easy_test_tok256_seq8.pt
[load] loaded GRPO model from runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference ARC-Easy-test: 100%|██████████████████████████████████| 1188/1188 [05:03<00:00,  3.92it/s]

[save] inference results -> runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_inference_test_results.json

==================== GRPO INFERENCE DONE ====================
{
  "dataset": "ARC-Easy",
  "split": "test",
  "checkpoint": "runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_best.pt",
  "cache_file": "arc_cached_features/ARC-Easy_test_tok256_seq8.pt",
  "num_examples": 2376,
  "correct": 791,
  "loss": 1.3568283144873803,
  "accuracy": 0.33291245791245794,
  "accuracy_percent": 33.29124579124579,
  "cache_build_time_sec": 0.0,
  "cache_build_time_hms": "00:00:00",
  "inference_time_sec": 303.10316632315516,
  "inference_time_hms": "00:05:03",
  "time_per_example_sec": 0.12756867269493063,
  "examples_per_second": 7.8389151417402685
}
[summary saved] runs/hlcm_arc_cached_reflogits_fixed/ARC-Easy/grpo_inference_test_summary.json


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class GRPOConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Challenge",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_challenge_cached_reflogits_fixed"
    cache_dir: str = "arc_challenge_cached_features"
    ref_logits_dir: str = "arc_challenge_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "head_only" | "last_blocks" | "full"
    n_last_blocks: int = 1

    # loader / memory
    train_batch_size: int = 1
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT: epoch-based
    sft_epochs: int = 2
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO: epoch-based
    grpo_epochs: int = 1
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # policy / reward
    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # choice chunking
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    # also keep non-layer modules trainable (frontend, embeddings, etc.)
    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "head_only":
        raise ValueError(
            "finetune_mode='head_only' is invalid for this script because no separate trainable head exists. "
            "Use 'last_blocks' or 'full'."
        )
    elif mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# ARC RAW / NORMALIZATION
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: GRPOConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)
    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    for k0 in range(0, K, max(1, choice_chunk_size)):
        k1 = min(K, k0 + max(1, choice_chunk_size))

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_arc_hybrid_hlcm(dataset_name: str, cfg: GRPOConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    # --------------------------------------------------------
    # build/load cached features
    # --------------------------------------------------------
    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_arc(cfg, dataset_name)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer)
    test_rows = build_or_load_cached_split(cfg, dataset_name, "test", test_hf, conceptizer) if test_hf is not None else []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    # --------------------------------------------------------
    # model
    # --------------------------------------------------------
    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metadata_base = {
        "dataset": dataset_name,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "num_test_examples": len(test_rows),
    }

    # --------------------------------------------------------
    # SFT
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    # --------------------------------------------------------
    # GRPO using cached ref logits
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    final_test = evaluate_hlcm(hlcm, test_loader, mu, sigma, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "final_test_loss": None if final_test is None else final_test["loss"],
            "final_test_acc": None if final_test is None else final_test["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "ref_logits_path": ref_logits_path,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] dataset={dataset_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = GRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_arc_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=1, bs_eval=2, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
choice_chunk_size=1

==================== ARC-Challenge ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building ARC-Challenge / train


cache:ARC-Challenge:train: 100%|████████████████████████████████| 1119/1119 [02:04<00:00,  9.02it/s]


...

[SFT][epoch 2/2] train_loss=1.3875 train_acc=0.2601 eval_loss=1.3871 eval_acc=0.2375
  saved sft_best.pt
[SFT][FINAL] best_acc=0.2375 total_train_time=34.08 min
[ref_logits] building arc_challenge_cached_ref_logits/ARC-Challenge_train_ref_logits_tok256_seq8.pt


precompute_ref_logits: 100%|██████████████████████████████████████| 560/560 [02:25<00:00,  3.85it/s]


[ref_logits] saved arc_challenge_cached_ref_logits/ARC-Challenge_train_ref_logits_tok256_seq8.pt (1119 rows)

========== ARC-Challenge :: STAGE 2 / GRPO ==========
[GRPO][BASE] loss=1.3871 acc=0.2375


grpo epoch 1/1: 100%|█| 1119/1119 [15:46<00:00,  1.18it/s, acc=0.2636, kl=0.0039, loss=-0.0024, lr=0


[GRPO][epoch 1/1] train_loss=-0.0024 train_acc=0.2636 eval_loss=1.3870 eval_acc=0.2308
[GRPO][FINAL] best_acc=0.2375 total_train_time=16.76 min
[HYBRID][FINAL] dataset=ARC-Challenge sft_best_acc=0.2375 grpo_best_acc=0.2375 final_eval_acc=0.2375 final_test_acc=0.2415 max_gpu_alloc=38605.1 MB saved -> runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge

All done. Outputs in: runs/hlcm_arc_challenge_cached_reflogits_fixed
Total wall time: 66.63 min


In [2]:

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Challenge",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_challenge_cached_reflogits_fixed"
    cache_dir: str = "arc_challenge_cached_features"

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# ARC DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        alpha_to_num = {"A": "1", "B": "2", "C": "3", "D": "4", "E": "5"}
        num_to_alpha = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}

        alt = alpha_to_num.get(key, num_to_alpha.get(key, None))
        if alt is not None and alt in labels:
            return labels.index(alt)

    return 0


def load_arc(cfg: EvalConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHED FEATURES
# ============================================================

def cache_file_path(cfg: EvalConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: EvalConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Base HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] base HLCM from {cfg.ckpt_path}")
    print(f"[load] base missing keys: {len(missing)}")
    print(f"[load] base unexpected keys: {len(unexpected)}")

    return model


def load_grpo_last_checkpoint(model: HyperbolicLCM, grpo_path: str, device: torch.device):
    if not os.path.exists(grpo_path):
        raise FileNotFoundError(f"grpo_last.pt not found: {grpo_path}")

    obj = torch.load(grpo_path, map_location="cpu")

    if isinstance(obj, dict) and "model" in obj:
        state = obj["model"]
    elif isinstance(obj, dict) and "model_state" in obj:
        state = obj["model_state"]
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] GRPO LAST checkpoint from {grpo_path}")
    print(f"[load] GRPO missing keys: {len(missing)}")
    print(f"[load] GRPO unexpected keys: {len(unexpected)}")

    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    info = {}

    if isinstance(obj, dict):
        for key in ["stage", "epoch", "best_eval_acc", "global_opt_step", "dataset"]:
            if key in obj:
                info[key] = obj[key]

    return info


# ============================================================
# LOGITS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk = max(1, int(choice_chunk_size))

    for k0 in range(0, K, chunk):
        k1 = min(K, k0 + chunk)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq_error = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins

        if b == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()

            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
    
# ============================================================
# RANKING METRICS
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics_hlcm(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4, 5),
) -> Dict[str, float]:
    model.eval()

    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}

    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        batch_size = labels.size(0)
        total += batch_size
        total_loss += float(loss.item()) * batch_size

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        total_brier += float(
            multiclass_brier_score(probs, labels, choice_mask).sum().item()
        )

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0

                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    if all_confidences:
        all_confidences = torch.cat(all_confidences)
        all_correctness = torch.cat(all_correctness)
        ece, mce = expected_calibration_error(
            all_confidences,
            all_correctness,
            n_bins=15,
        )
    else:
        ece, mce = 0.0, 0.0

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results

# ============================================================
# EVAL ONE DATASET
# ============================================================

def eval_one_dataset(dataset_name: str, cfg: EvalConfig, device: torch.device):
    print(f"\n==================== EVAL ONLY: {dataset_name} ====================")

    dataset_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    grpo_last_path = os.path.join(dataset_dir, "grpo_last.pt")

    eval_hf, test_hf = load_arc(cfg, dataset_name)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    test_rows = []
    if test_hf is not None:
        test_rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=dataset_name,
            split_name="test",
            hf_split=test_hf,
            conceptizer=conceptizer,
        )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_loader = DataLoader(
            CachedMCQDataset(test_rows),
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    model = load_base_hlcm(cfg, device)
    ckpt_info = load_grpo_last_checkpoint(model, grpo_last_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    validation_metrics = evaluate_ranking_metrics_hlcm(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4, 5),
    )

    test_metrics = None

    if test_loader is not None:
        test_metrics = evaluate_ranking_metrics_hlcm(
            model=model,
            loader=test_loader,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            k_values=(1, 2, 3, 4, 5),
        )

    summary = {
        "dataset": dataset_name,
        "checkpoint": grpo_last_path,
        "checkpoint_info": ckpt_info,
        "num_validation_examples": len(eval_rows),
        "num_test_examples": len(test_rows),
        "validation": validation_metrics,
        "test": test_metrics,
    }

    ensure_dir(dataset_dir)

    save_path = os.path.join(dataset_dir, "grpo_last_eval_precision_recall_ranking_only.json")

    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    del model
    cuda_cleanup()

    return summary


# ============================================================
# RUN EVAL ONLY
# ============================================================

def main():
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("This script is EVAL ONLY.")

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    all_results = {}

    for dataset_name in cfg.datasets_to_run:
        all_results[dataset_name] = eval_one_dataset(dataset_name, cfg, device)

    save_path = os.path.join(cfg.out_dir, "grpo_last_eval_precision_recall_ranking_only_all.json")

    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_results, indent=2))
    print(f"[saved] {save_path}")


if __name__ == "__main__":
    main()

Device: cpu
This script is EVAL ONLY.

==================== EVAL ONLY: ARC-Challenge ====================


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Challenge' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Challenge/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:31:06 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading arc_challenge_cached_features/ARC-Challenge_validation_tok256_seq8.pt
[cache] loading arc_challenge_cached_features/ARC-Challenge_test_tok256_seq8.pt
[load] base HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] base missing keys: 0
[load] base unexpected keys: 0
[load] GRPO LAST checkpoint from runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt
[load] GRPO missing keys: 0
[load] GRPO unexpected keys: 0


{
  "dataset": "ARC-Challenge",
  "checkpoint": "runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt",
  "checkpoint_info": {
    "stage": "grpo_last",
    "epoch": 1,
    "global_opt_step": 140,
    "dataset": "ARC-Challenge"
  },
  "num_validation_examples": 299,
  "num_test_examples": 1172,
  "validation": {
    "loss": 1.3870431555553424,
    "accuracy": 0.23076923076923078,
    "brier_score": 0.7507776883134873,
    "ece": 0.04638226496510445,
    "mce": 0.07817557454109192,
    "ece_bins": 15,
    "precision@1": 0.23076923076923078,
    "recall@1": 0.23076923076923078,
    "precision@2": 0.2408026755852843,
    "recall@2": 0.4816053511705686,
    "precision@3": 0.24191750278706817,
    "recall@3": 0.725752508361204,
    "precision@4": 0.2508361204013378,
    "recall@4": 1.0,
    "precision@5": 0.2506688963210703,
    "recall@5": 1.0,
    "mrr": 0.5061315496098104
  },
  "test": {
    "loss": 1.385781181753699,
    "accuracy": 0.24829351535836178,
    "brier_

In [3]:
#Inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "ARC-Challenge"
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/hlcm_arc_challenge_cached_reflogits_fixed"
    cache_dir: str = "arc_challenge_cached_features"

    # Your GRPO checkpoint
    grpo_ckpt_path: str = "runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt"

    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0

    # use "validation" or "test"
    split: str = "test"

    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# ARC DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: InferenceConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})

    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])

    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHE FEATURES
# ============================================================

def cache_file_path(cfg: InferenceConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(f"Cache not found: {path}")

    if conceptizer is None:
        raise RuntimeError("Conceptizer required because cache is missing.")

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": choices,
                    "cmask": cmask,
                    "choice_mask": choice_mask,
                    "label": int(label),
                    "num_choices": int(K),
                }
            )

        except Exception as e:
            skipped += 1
            print(
                f"[warn] skipped one example in {dataset_name}-{split_name}: "
                f"{type(e).__name__}: {e}"
            )

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]

        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_hlcm(cfg: InferenceConfig, device: torch.device) -> HyperbolicLCM:
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(f"GRPO checkpoint not found: {cfg.grpo_ckpt_path}")

    model = build_hlcm_from_cfg(cfg).to(device)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_last.pt does not contain key 'model'.")

    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded GRPO model from {cfg.grpo_ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape

    out = torch.empty(
        (B, D),
        device=h_tan.device,
        dtype=h_tan.dtype,
    )

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(
        model=model,
        x=q,
        pad_mask=qmask,
        mu=mu,
        sigma=sigma,
    )
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []

    for k0 in range(0, K, max(1, cfg.choice_chunk_size)):
        k1 = min(K, k0 + max(1, cfg.choice_chunk_size))

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(
            model=model,
            x=flat,
            pad_mask=flat_mask,
            mu=mu,
            sigma=sigma,
        )

        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits


# ============================================================
# INFERENCE + TIME
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
    save_path: Optional[str] = None,
) -> Dict[str, Any]:
    model.eval()

    device = next(model.parameters()).device

    total = 0
    correct = 0
    predictions = []
    total_loss = 0.0

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference {cfg.dataset_name}-{cfg.split}"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)

        total_loss += float(loss.item()) * bs
        correct += int((preds == labels).sum().item())
        total += int(bs)

        for i in range(bs):
            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(preds[i].item() == labels[i].item()),
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "dataset": cfg.dataset_name,
        "split": cfg.split,
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_arc(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[dataset]", cfg.dataset_name)
    print("[split]", cfg.split)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    train_hf, eval_hf, test_hf = load_arc(cfg, cfg.dataset_name)

    if cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    elif cfg.split in {"validation", "val", "dev"}:
        hf_split = eval_hf
        split_name = "validation"
    elif cfg.split == "test":
        if test_hf is None:
            raise RuntimeError("No test split available.")
        hf_split = test_hf
        split_name = "test"
    else:
        raise ValueError("cfg.split must be one of: train, validation, test")

    cache_path = cache_file_path(cfg, cfg.dataset_name, split_name)

    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer_device = torch.device(cfg.conceptizer_device)
        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=conceptizer_device,
        )

        cache_t0 = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=conceptizer,
        )

        cache_build_time_sec = time.perf_counter() - cache_t0

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=None,
        )

    print(f"[data] examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model = load_grpo_hlcm(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    save_path = os.path.join(
        dataset_out_dir,
        f"grpo_last_inference_{split_name}_results.json",
    )

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "dataset": cfg.dataset_name,
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"grpo_last_inference_{split_name}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== GRPO LAST INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [4]:
cfg = InferenceConfig(
    dataset_name="ARC-Challenge",

    grpo_ckpt_path="runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt",
    normalizer_path="normalizer.pt",

    out_dir="runs/hlcm_arc_challenge_cached_reflogits_fixed",
    cache_dir="arc_challenge_cached_features",

    split="test",
    eval_batch_size=2,
    choice_chunk_size=1,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_arc(cfg)

[device] cuda:0
[dataset] ARC-Challenge
[split] test


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Challenge' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Challenge/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:31:06 2026).


[cache] loading arc_challenge_cached_features/ARC-Challenge_test_tok256_seq8.pt
[data] examples=1172
[cache] arc_challenge_cached_features/ARC-Challenge_test_tok256_seq8.pt
[load] loaded GRPO model from runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference ARC-Challenge-test: 100%|███████████████████████████████| 586/586 [02:26<00:00,  4.00it/s]


[save] inference results -> runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last_inference_test_results.json

==================== GRPO LAST INFERENCE DONE ====================
{
  "dataset": "ARC-Challenge",
  "split": "test",
  "checkpoint": "runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last.pt",
  "cache_file": "arc_challenge_cached_features/ARC-Challenge_test_tok256_seq8.pt",
  "num_examples": 1172,
  "correct": 291,
  "loss": 1.3857802705960063,
  "accuracy": 0.24829351535836178,
  "accuracy_percent": 24.829351535836178,
  "cache_build_time_sec": 0.0,
  "cache_build_time_hms": "00:00:00",
  "inference_time_sec": 146.5141604221426,
  "inference_time_hms": "00:02:26",
  "time_per_example_sec": 0.12501208227145272,
  "examples_per_second": 7.999226809362219
}
[summary saved] runs/hlcm_arc_challenge_cached_reflogits_fixed/ARC-Challenge/grpo_last_inference_test_summary.json


In [5]:
print("Accuracy:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

Accuracy: 24.829351535836178
Inference time: 146.5141604221426
Time/example: 0.12501208227145272
Examples/sec: 7.999226809362219


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class GRPOConfig:
    datasets_to_run: Tuple[str, ...] = ("main",)   # OpenBookQA config
    openbookqa_dataset_name: str = "allenai/openbookqa"

    out_dir: str = "runs/hlcm_openbookqa"
    cache_dir: str = "openbookqa_cached_features"
    ref_logits_dir: str = "openbookqa_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 1

    # loader / memory
    train_batch_size: int = 1
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT: epoch-based
    sft_epochs: int = 2
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO: epoch-based
    grpo_epochs: int = 1
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # policy / reward
    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # choice chunking
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "head_only":
        raise ValueError(
            "finetune_mode='head_only' is invalid for this script because no separate trainable head exists. "
            "Use 'last_blocks' or 'full'."
        )
    elif mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# OPENBOOKQA RAW / NORMALIZATION
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_openbookqa(cfg: GRPOConfig, subset_name: str):
    raw = load_dataset(cfg.openbookqa_dataset_name, subset_name)
    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return train_split, eval_split, test_split


def normalize_openbookqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question_stem", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_openbookqa_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, choice_chunk_size)
    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_openbookqa_hybrid_hlcm(dataset_name: str, cfg: GRPOConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    # --------------------------------------------------------
    # build/load cached features
    # --------------------------------------------------------
    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_openbookqa(cfg, dataset_name)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer)
    test_rows = build_or_load_cached_split(cfg, dataset_name, "test", test_hf, conceptizer) if test_hf is not None else []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    # --------------------------------------------------------
    # model
    # --------------------------------------------------------
    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metadata_base = {
        "dataset": dataset_name,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "num_test_examples": len(test_rows),
    }

    # --------------------------------------------------------
    # SFT
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    # --------------------------------------------------------
    # GRPO using cached ref logits
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    final_test = evaluate_hlcm(hlcm, test_loader, mu, sigma, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "final_test_loss": None if final_test is None else final_test["loss"],
            "final_test_acc": None if final_test is None else final_test["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "ref_logits_path": ref_logits_path,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] dataset={dataset_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = GRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_openbookqa_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=1, bs_eval=2, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
choice_chunk_size=1

==================== main ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building main / train


cache:main:train: 100%|█████████████████████████████████████████| 4957/4957 [08:20<00:00,  9.91it/s]


...

[SFT][epoch 2/2] train_loss=1.3642 train_acc=0.3216 eval_loss=1.3622 eval_acc=0.3020
  saved sft_best.pt
[SFT][FINAL] best_acc=0.3020 total_train_time=142.72 min
[ref_logits] building openbookqa_cached_ref_logits/main_train_ref_logits_tok256_seq8.pt


precompute_ref_logits: 100%|████████████████████████████████████| 2479/2479 [10:42<00:00,  3.86it/s]


[ref_logits] saved openbookqa_cached_ref_logits/main_train_ref_logits_tok256_seq8.pt (4957 rows)

========== main :: STAGE 2 / GRPO ==========
[GRPO][BASE] loss=1.3622 acc=0.3020


grpo epoch 1/1: 100%|█| 4957/4957 [1:08:20<00:00,  1.21it/s, acc=0.3514, kl=0.0038, loss=-0.0210, lr


[GRPO][epoch 1/1] train_loss=-0.0210 train_acc=0.3514 eval_loss=1.4115 eval_acc=0.3060
  saved grpo_best.pt
[GRPO][FINAL] best_acc=0.3060 total_train_time=70.03 min
[HYBRID][FINAL] dataset=main sft_best_acc=0.3020 grpo_best_acc=0.3060 final_eval_acc=0.3060 final_test_acc=0.3080 max_gpu_alloc=34228.4 MB saved -> runs/hlcm_openbookqa/main

All done. Outputs in: runs/hlcm_openbookqa
Total wall time: 241.89 min


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    datasets_to_run: Tuple[str, ...] = ("main",)
    openbookqa_dataset_name: str = "allenai/openbookqa"

    out_dir: str = "runs/hlcm_openbookqa"
    cache_dir: str = "openbookqa_cached_features"

    # pretrained HLCM checkpoint
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM architecture
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # eval
    eval_batch_size: int = 2
    num_workers: int = 0

    # scoring
    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# OPENBOOKQA DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_openbookqa(cfg: EvalConfig, subset_name: str):
    raw = load_dataset(cfg.openbookqa_dataset_name, subset_name)
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return eval_split, test_split


def normalize_openbookqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question_stem", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: EvalConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: EvalConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_openbookqa_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Base HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] base HLCM from {cfg.ckpt_path}")
    print(f"[load] base missing keys: {len(missing)}")
    print(f"[load] base unexpected keys: {len(unexpected)}")

    return model


def load_grpo_best_checkpoint(model: HyperbolicLCM, grpo_path: str, device: torch.device):
    if not os.path.exists(grpo_path):
        raise FileNotFoundError(f"grpo_best.pt not found: {grpo_path}")

    obj = torch.load(grpo_path, map_location="cpu")

    if isinstance(obj, dict) and "model" in obj:
        state = obj["model"]
    elif isinstance(obj, dict) and "model_state" in obj:
        state = obj["model_state"]
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] GRPO BEST checkpoint from {grpo_path}")
    print(f"[load] GRPO missing keys: {len(missing)}")
    print(f"[load] GRPO unexpected keys: {len(unexpected)}")

    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    info = {}
    if isinstance(obj, dict):
        for key in ["stage", "epoch", "best_eval_acc", "global_opt_step", "dataset"]:
            if key in obj:
                info[key] = obj[key]

    return info


# ============================================================
# LOGITS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, int(choice_chunk_size))

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)

    sq_error = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins

        if b == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()

            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
# ============================================================
# RANKING METRICS
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics_hlcm(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4),
) -> Dict[str, float]:
    model.eval()

    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0
    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        batch_size = labels.size(0)
        total += batch_size
        total_loss += float(loss.item()) * batch_size

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        total_brier += float(
            multiclass_brier_score(probs, labels, choice_mask).sum().item()
        )

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    if all_confidences:
        all_confidences = torch.cat(all_confidences)
        all_correctness = torch.cat(all_correctness)
        ece, mce = expected_calibration_error(
            all_confidences,
            all_correctness,
            n_bins=15,
        )
    else:
        ece, mce = 0.0, 0.0

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


# ============================================================
# EVAL ONE DATASET
# ============================================================

def eval_one_dataset(dataset_name: str, cfg: EvalConfig, device: torch.device):
    print(f"\n==================== EVAL ONLY: {dataset_name} ====================")

    dataset_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    grpo_best_path = os.path.join(dataset_dir, "grpo_best.pt")

    eval_hf, test_hf = load_openbookqa(cfg, dataset_name)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    test_rows = []
    if test_hf is not None:
        test_rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=dataset_name,
            split_name="test",
            hf_split=test_hf,
            conceptizer=conceptizer,
        )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_loader = DataLoader(
            CachedMCQDataset(test_rows),
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    model = load_base_hlcm(cfg, device)
    ckpt_info = load_grpo_best_checkpoint(model, grpo_best_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    validation_metrics = evaluate_ranking_metrics_hlcm(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4),
    )

    test_metrics = None
    if test_loader is not None:
        test_metrics = evaluate_ranking_metrics_hlcm(
            model=model,
            loader=test_loader,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            k_values=(1, 2, 3, 4),
        )

    summary = {
        "dataset": dataset_name,
        "checkpoint": grpo_best_path,
        "checkpoint_info": ckpt_info,
        "num_validation_examples": len(eval_rows),
        "num_test_examples": len(test_rows),
        "validation": validation_metrics,
        "test": test_metrics,
    }

    ensure_dir(dataset_dir)
    save_path = os.path.join(dataset_dir, "grpo_best_eval_precision_recall_ranking_only.json")

    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    del model
    cuda_cleanup()

    return summary


# ============================================================
# MAIN
# ============================================================

def main():
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("This script is EVAL ONLY.")

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    all_results = {}

    for dataset_name in cfg.datasets_to_run:
        all_results[dataset_name] = eval_one_dataset(dataset_name, cfg, device)

    save_path = os.path.join(cfg.out_dir, "grpo_best_eval_precision_recall_ranking_only_all.json")

    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_results, indent=2))
    print(f"[saved] {save_path}")


if __name__ == "__main__":
    main()

Device: cuda:0
This script is EVAL ONLY.

==================== EVAL ONLY: main ====================


Using the latest cached version of the dataset since allenai/openbookqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/user/.cache/huggingface/datasets/allenai___openbookqa/main/0.0.0/388097ea7776314e93a529163e0fea805b8a6454 (last modified on Wed Mar 18 08:37:00 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading openbookqa_cached_features/main_validation_tok256_seq8.pt
[cache] loading openbookqa_cached_features/main_test_tok256_seq8.pt
[load] base HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] base missing keys: 0
[load] base unexpected keys: 0
[load] GRPO BEST checkpoint from runs/hlcm_openbookqa/main/grpo_best.pt
[load] GRPO missing keys: 0
[load] GRPO unexpected keys: 0


{
  "dataset": "main",
  "checkpoint": "runs/hlcm_openbookqa/main/grpo_best.pt",
  "checkpoint_info": {
    "stage": "grpo_best",
    "epoch": 1,
    "best_eval_acc": 0.306,
    "global_opt_step": 620,
    "dataset": "main"
  },
  "num_validation_examples": 500,
  "num_test_examples": 500,
  "validation": {
    "loss": 1.4114974603652954,
    "accuracy": 0.306,
    "brier_score": 0.7622450777292251,
    "ece": 0.10533864492353007,
    "mce": 0.5650937557220459,
    "ece_bins": 15,
    "precision@1": 0.306,
    "recall@1": 0.306,
    "precision@2": 0.276,
    "recall@2": 0.552,
    "precision@3": 0.2619999999999987,
    "recall@3": 0.786,
    "precision@4": 0.25,
    "recall@4": 1.0,
    "mrr": 0.5605000000000006
  },
  "test": {
    "loss": 1.4577924807071685,
    "accuracy": 0.308,
    "brier_score": 0.7788246488571167,
    "ece": 0.09043784530061154,
    "mce": 0.8124259114265442,
    "ece_bins": 15,
    "precision@1": 0.308,
    "recall@1": 0.308,
    "precision@2": 0.261,
    "reca

In [6]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "main"
    openbookqa_dataset_name: str = "allenai/openbookqa"

    out_dir: str = "runs/hlcm_openbookqa"
    cache_dir: str = "openbookqa_cached_features"

    grpo_ckpt_path: str = "runs/hlcm_openbookqa/main/grpo_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0

    split: str = "test"  # "validation" or "test"
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# OPENBOOKQA DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_openbookqa(cfg: InferenceConfig, subset_name: str):
    raw = load_dataset(cfg.openbookqa_dataset_name, subset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_openbookqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question_stem", "")
    choices = ex.get("choices", {})

    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])

    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# CACHE FEATURES
# ============================================================

def cache_file_path(cfg: InferenceConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(f"Cache not found: {path}")

    if conceptizer is None:
        raise RuntimeError("Conceptizer required because cache is missing.")

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_openbookqa_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": choices,
                    "cmask": cmask,
                    "choice_mask": choice_mask,
                    "label": int(label),
                    "num_choices": int(K),
                }
            )

        except Exception as e:
            skipped += 1
            print(
                f"[warn] skipped one example in {dataset_name}-{split_name}: "
                f"{type(e).__name__}: {e}"
            )

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]

        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_hlcm(cfg: InferenceConfig, device: torch.device) -> HyperbolicLCM:
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(f"GRPO checkpoint not found: {cfg.grpo_ckpt_path}")

    model = build_hlcm_from_cfg(cfg).to(device)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_best.pt does not contain key 'model'.")

    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded GRPO model from {cfg.grpo_ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(
        model=model,
        x=q,
        pad_mask=qmask,
        mu=mu,
        sigma=sigma,
    )
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(
            model=model,
            x=flat,
            pad_mask=flat_mask,
            mu=mu,
            sigma=sigma,
        )

        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits


# ============================================================
# INFERENCE + TIME
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
    save_path: Optional[str] = None,
) -> Dict[str, Any]:
    model.eval()
    device = next(model.parameters()).device

    total = 0
    correct = 0
    total_loss = 0.0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference OpenBookQA-{cfg.split}"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)

        total_loss += float(loss.item()) * bs
        correct += int((preds == labels).sum().item())
        total += int(bs)

        for i in range(bs):
            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(preds[i].item() == labels[i].item()),
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "dataset": cfg.dataset_name,
        "split": cfg.split,
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_openbookqa(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[dataset]", cfg.dataset_name)
    print("[split]", cfg.split)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    train_hf, eval_hf, test_hf = load_openbookqa(cfg, cfg.dataset_name)

    if cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    elif cfg.split in {"validation", "val", "dev"}:
        hf_split = eval_hf
        split_name = "validation"
    elif cfg.split == "test":
        if test_hf is None:
            raise RuntimeError("No test split available.")
        hf_split = test_hf
        split_name = "test"
    else:
        raise ValueError("cfg.split must be one of: train, validation, test")

    cache_path = cache_file_path(cfg, cfg.dataset_name, split_name)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer_device = torch.device(cfg.conceptizer_device)
        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=conceptizer_device,
        )

        cache_t0 = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=conceptizer,
        )

        cache_build_time_sec = time.perf_counter() - cache_t0

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=cfg.dataset_name,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=None,
        )

    print(f"[data] examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model = load_grpo_hlcm(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    save_path = os.path.join(
        dataset_out_dir,
        f"grpo_best_inference_{split_name}_results.json",
    )

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "dataset": cfg.dataset_name,
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"grpo_best_inference_{split_name}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== GRPO BEST INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [7]:
cfg = InferenceConfig(
    dataset_name="main",

    grpo_ckpt_path="runs/hlcm_openbookqa/main/grpo_best.pt",
    normalizer_path="normalizer.pt",

    out_dir="runs/hlcm_openbookqa",
    cache_dir="openbookqa_cached_features",

    split="test",
    eval_batch_size=2,
    choice_chunk_size=1,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_openbookqa(cfg)

[device] cuda:0
[dataset] main
[split] test


Using the latest cached version of the dataset since allenai/openbookqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/user/.cache/huggingface/datasets/allenai___openbookqa/main/0.0.0/388097ea7776314e93a529163e0fea805b8a6454 (last modified on Wed Mar 18 08:37:00 2026).


[cache] not found: openbookqa_cached_features/main_test_tok256_seq8.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building main / test


cache:main:test: 100%|████████████████████████████████████████████| 500/500 [00:53<00:00,  9.41it/s]


[cache] saved openbookqa_cached_features/main_test_tok256_seq8.pt (500 examples, skipped=0)
[data] examples=500
[cache] openbookqa_cached_features/main_test_tok256_seq8.pt
[load] loaded GRPO model from runs/hlcm_openbookqa/main/grpo_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference OpenBookQA-test: 100%|██████████████████████████████████| 250/250 [01:04<00:00,  3.85it/s]


[save] inference results -> runs/hlcm_openbookqa/main/grpo_best_inference_test_results.json

==================== GRPO BEST INFERENCE DONE ====================
{
  "dataset": "main",
  "split": "test",
  "checkpoint": "runs/hlcm_openbookqa/main/grpo_best.pt",
  "cache_file": "openbookqa_cached_features/main_test_tok256_seq8.pt",
  "num_examples": 500,
  "correct": 154,
  "loss": 1.4577924807071685,
  "accuracy": 0.308,
  "accuracy_percent": 30.8,
  "cache_build_time_sec": 53.212378611788154,
  "cache_build_time_hms": "00:00:53",
  "inference_time_sec": 65.06051831506193,
  "inference_time_hms": "00:01:05",
  "time_per_example_sec": 0.13012103663012387,
  "examples_per_second": 7.685152423451363
}
[summary saved] runs/hlcm_openbookqa/main/grpo_best_inference_test_summary.json


In [8]:
print("Accuracy:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

Accuracy: 30.8
Inference time: 65.06051831506193
Time/example: 0.13012103663012387
Examples/sec: 7.685152423451363


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class GRPOConfig:
    datasets_to_run: Tuple[str, ...] = ("default",)   # CommonsenseQA config
    commonsenseqa_dataset_name: str = "tau/commonsense_qa"

    out_dir: str = "runs/hlcm_commonsenseqa_cached_reflogits_fixed"
    cache_dir: str = "commonsenseqa_cached_features"
    ref_logits_dir: str = "commonsenseqa_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 1

    # loader / memory
    train_batch_size: int = 1
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT: epoch-based
    sft_epochs: int = 2
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO: epoch-based
    grpo_epochs: int = 1
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # policy / reward
    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # choice chunking
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "head_only":
        raise ValueError(
            "finetune_mode='head_only' is invalid for this script because no separate trainable head exists. "
            "Use 'last_blocks' or 'full'."
        )
    elif mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# COMMONSENSEQA RAW / NORMALIZATION
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_commonsenseqa(cfg: GRPOConfig, subset_name: str):
    raw = load_dataset(cfg.commonsenseqa_dataset_name, subset_name)
    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return train_split, eval_split, test_split


def normalize_commonsenseqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    question_concept = ex.get("question_concept", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    # Include concept in the prompt because CommonsenseQA exposes it explicitly.
    if isinstance(question_concept, str) and question_concept.strip():
        q_text = f"Question: {str(q).strip()}\nConcept: {question_concept.strip()}"
    else:
        q_text = str(q)

    return q_text, clean_choices, y


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_commonsenseqa_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, choice_chunk_size)
    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_commonsenseqa_hybrid_hlcm(dataset_name: str, cfg: GRPOConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    # --------------------------------------------------------
    # build/load cached features
    # --------------------------------------------------------
    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_commonsenseqa(cfg, dataset_name)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer)
    test_rows = build_or_load_cached_split(cfg, dataset_name, "test", test_hf, conceptizer) if test_hf is not None else []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    # --------------------------------------------------------
    # model
    # --------------------------------------------------------
    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metadata_base = {
        "dataset": dataset_name,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "num_test_examples": len(test_rows),
    }

    # --------------------------------------------------------
    # SFT
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    # --------------------------------------------------------
    # GRPO using cached ref logits
    # --------------------------------------------------------
    print(f"\n========== {dataset_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    final_test = evaluate_hlcm(hlcm, test_loader, mu, sigma, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "final_test_loss": None if final_test is None else final_test["loss"],
            "final_test_acc": None if final_test is None else final_test["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "ref_logits_path": ref_logits_path,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] dataset={dataset_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = GRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_commonsenseqa_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=1, bs_eval=2, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
choice_chunk_size=1

==================== default ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building default / train


cache:default:train: 100%|██████████████████████████████████████| 9741/9741 [17:53<00:00,  9.08it/s]


...

[SFT][epoch 2/2] train_loss=1.5498 train_acc=0.3091 eval_loss=1.8931 eval_acc=0.3022
[SFT][FINAL] best_acc=0.3350 total_train_time=336.37 min
[ref_logits] building commonsenseqa_cached_ref_logits/default_train_ref_logits_tok256_seq8.pt


precompute_ref_logits: 100%|████████████████████████████████████| 4871/4871 [25:14<00:00,  3.22it/s]


[ref_logits] saved commonsenseqa_cached_ref_logits/default_train_ref_logits_tok256_seq8.pt (9741 rows)

========== default :: STAGE 2 / GRPO ==========
[GRPO][BASE] loss=1.5931 acc=0.3350


grpo epoch 1/1: 100%|█| 9741/9741 [2:47:07<00:00,  1.03s/it, acc=0.3049, kl=0.0472, loss=-0.0353, lr


[GRPO][epoch 1/1] train_loss=-0.0353 train_acc=0.3049 eval_loss=2.0489 eval_acc=0.3170
[GRPO][FINAL] best_acc=0.3350 total_train_time=170.68 min
[HYBRID][FINAL] dataset=default sft_best_acc=0.3350 grpo_best_acc=0.3350 final_eval_acc=0.3350 final_test_acc=0.1886 max_gpu_alloc=38605.1 MB saved -> runs/hlcm_commonsenseqa_cached_reflogits_fixed/default

All done. Outputs in: runs/hlcm_commonsenseqa_cached_reflogits_fixed
Total wall time: 570.97 min


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    datasets_to_run: Tuple[str, ...] = ("default",)
    commonsenseqa_dataset_name: str = "tau/commonsense_qa"

    out_dir: str = "runs/hlcm_commonsenseqa_cached_reflogits_fixed"
    cache_dir: str = "commonsenseqa_cached_features"

    # Base pretrained HLCM checkpoint.
    # final_hybrid.pt will overwrite this model state after model creation.
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM architecture
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # eval
    eval_batch_size: int = 2
    num_workers: int = 0

    # scoring
    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# COMMONSENSEQA DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_commonsenseqa(cfg: EvalConfig, subset_name: str):
    raw = load_dataset(cfg.commonsenseqa_dataset_name, subset_name)

    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return eval_split, test_split


def normalize_commonsenseqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    question_concept = ex.get("question_concept", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    if isinstance(question_concept, str) and question_concept.strip():
        q_text = f"Question: {str(q).strip()}\nConcept: {question_concept.strip()}"
    else:
        q_text = str(q)

    return q_text, clean_choices, y


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: EvalConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: EvalConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_commonsenseqa_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Base HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] base HLCM from {cfg.ckpt_path}")
    print(f"[load] base missing keys: {len(missing)}")
    print(f"[load] base unexpected keys: {len(unexpected)}")

    return model


def load_final_hybrid_checkpoint(model: HyperbolicLCM, final_path: str, device: torch.device):
    if not os.path.exists(final_path):
        raise FileNotFoundError(f"final_hybrid.pt not found: {final_path}")

    obj = torch.load(final_path, map_location="cpu")

    if isinstance(obj, dict) and "model" in obj:
        state = obj["model"]
    elif isinstance(obj, dict) and "model_state" in obj:
        state = obj["model_state"]
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] FINAL HYBRID checkpoint from {final_path}")
    print(f"[load] final_hybrid missing keys: {len(missing)}")
    print(f"[load] final_hybrid unexpected keys: {len(unexpected)}")

    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    info = {}

    if isinstance(obj, dict):
        for key in [
            "stage",
            "sft_best_acc",
            "grpo_best_acc",
            "final_eval_loss",
            "final_eval_acc",
            "final_test_loss",
            "final_test_acc",
            "dataset",
        ]:
            if key in obj:
                info[key] = obj[key]

    return info


# ============================================================
# LOGITS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, int(choice_chunk_size))

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq_error = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins

        if b == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()

            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
    
# ============================================================
# RANKING METRICS
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics_hlcm(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4, 5),
) -> Dict[str, float]:
    model.eval()

    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0
    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        batch_size = labels.size(0)
        total += batch_size
        total_loss += float(loss.item()) * batch_size

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        total_brier += float(
            multiclass_brier_score(probs, labels, choice_mask).sum().item()
        )

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    if all_confidences:
        confidences_cat = torch.cat(all_confidences)
        correctness_cat = torch.cat(all_correctness)
        ece, mce = expected_calibration_error(
            confidences_cat,
            correctness_cat,
            n_bins=15,
        )
    else:
        ece, mce = 0.0, 0.0

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


# ============================================================
# EVAL ONE DATASET
# ============================================================

def eval_one_dataset(dataset_name: str, cfg: EvalConfig, device: torch.device):
    print(f"\n==================== EVAL ONLY: {dataset_name} ====================")

    dataset_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    final_hybrid_path = os.path.join(dataset_dir, "final_hybrid.pt")

    eval_hf, test_hf = load_commonsenseqa(cfg, dataset_name)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    test_rows = []
    if test_hf is not None:
        test_rows = build_or_load_cached_split(
            cfg=cfg,
            dataset_name=dataset_name,
            split_name="test",
            hf_split=test_hf,
            conceptizer=conceptizer,
        )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_loader = DataLoader(
            CachedMCQDataset(test_rows),
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    model = load_base_hlcm(cfg, device)
    ckpt_info = load_final_hybrid_checkpoint(model, final_hybrid_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    validation_metrics = evaluate_ranking_metrics_hlcm(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4, 5),
    )

    test_metrics = None
    if test_loader is not None:
        test_metrics = evaluate_ranking_metrics_hlcm(
            model=model,
            loader=test_loader,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            k_values=(1, 2, 3, 4, 5),
        )

    summary = {
        "dataset": dataset_name,
        "checkpoint": final_hybrid_path,
        "checkpoint_info": ckpt_info,
        "num_validation_examples": len(eval_rows),
        "num_test_examples": len(test_rows),
        "validation": validation_metrics,
        "test": test_metrics,
    }

    ensure_dir(dataset_dir)

    save_path = os.path.join(dataset_dir, "final_hybrid_eval_precision_recall_ranking_only.json")

    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    del model
    cuda_cleanup()

    return summary


# ============================================================
# RUN EVAL ONLY
# ============================================================

def main():
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    all_results = {}

    for dataset_name in cfg.datasets_to_run:
        all_results[dataset_name] = eval_one_dataset(dataset_name, cfg, device)

    save_path = os.path.join(cfg.out_dir, "final_hybrid_eval_precision_recall_ranking_only_all.json")

    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_results, indent=2))
    print(f"[saved] {save_path}")


if __name__ == "__main__":
    main()

Device: cuda:0

==================== EVAL ONLY: default ====================


Using the latest cached version of the dataset since tau/commonsense_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/user/.cache/huggingface/datasets/tau___commonsense_qa/default/0.0.0/94630fe30dad47192a8546eb75f094926d47e155 (last modified on Fri Mar 20 18:23:31 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading commonsenseqa_cached_features/default_validation_tok256_seq8.pt
[cache] loading commonsenseqa_cached_features/default_test_tok256_seq8.pt
[load] base HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] base missing keys: 0
[load] base unexpected keys: 0
[load] FINAL HYBRID checkpoint from runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt
[load] final_hybrid missing keys: 0
[load] final_hybrid unexpected keys: 0


{
  "dataset": "default",
  "checkpoint": "runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt",
  "checkpoint_info": {
    "stage": "final_hybrid",
    "sft_best_acc": 0.33497133497133497,
    "grpo_best_acc": 0.33497133497133497,
    "final_eval_loss": 1.5930794524814533,
    "final_eval_acc": 0.33497133497133497,
    "final_test_loss": 1.985058375147351,
    "final_test_acc": 0.18859649122807018,
    "dataset": "default"
  },
  "num_validation_examples": 1221,
  "num_test_examples": 1140,
  "validation": {
    "loss": 1.5930794556056935,
    "accuracy": 0.33497133497133497,
    "brier_score": 0.7891987521417995,
    "ece": 0.09721148352600434,
    "mce": 0.39612460136413574,
    "ece_bins": 15,
    "precision@1": 0.33415233415233414,
    "recall@1": 0.33415233415233414,
    "precision@2": 0.2796887796887797,
    "recall@2": 0.5593775593775594,
    "precision@3": 0.24897624897624912,
    "recall@3": 0.7469287469287469,
    "precision@4": 0.22563472563472564,
    "r

In [9]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "default"
    commonsenseqa_dataset_name: str = "tau/commonsense_qa"

    out_dir: str = "runs/hlcm_commonsenseqa_cached_reflogits_fixed"
    cache_dir: str = "commonsenseqa_cached_features"

    final_hybrid_path: str = "runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0

    split: str = "validation"
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds):
    seconds = int(max(0, seconds))
    return f"{seconds//3600:02d}:{(seconds%3600)//60:02d}:{seconds%60:02d}"


def pick_device(prefer_gpu_index=0):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device):
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(path, device):
    if not path or not os.path.exists(path):
        print("[normalizer] not found")
        return None, None
    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print("[normalizer] loaded")
    return mu, sigma


# ============================================================
# READ grpo_best_acc FROM final_hybrid.pt
# ============================================================

def read_grpo_best_acc(final_hybrid_path):
    if not os.path.exists(final_hybrid_path):
        raise FileNotFoundError(final_hybrid_path)

    obj = torch.load(final_hybrid_path, map_location="cpu")

    print("Available keys:")
    print(list(obj.keys()))

    grpo_best_acc = obj.get("grpo_best_acc", None)

    if grpo_best_acc is None:
        print("\n[warning] grpo_best_acc not found in checkpoint.")
    else:
        print(f"\ngrpo_best_acc = {grpo_best_acc:.6f}")
        print(f"grpo_best_acc (%) = {grpo_best_acc * 100:.2f}%")

    return grpo_best_acc, obj


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts):
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text):
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i+self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break
        return chunks[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text):
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i+self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# COMMONSENSEQA DATA
# ============================================================

def answerkey_to_index(answer_key, labels):
    if not isinstance(answer_key, str):
        return 0
    key = answer_key.strip()
    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_commonsenseqa(cfg):
    raw = load_dataset(cfg.commonsenseqa_dataset_name, cfg.dataset_name)
    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None
    return train_split, eval_split, test_split


def normalize_commonsenseqa_example(ex, min_valid_choices):
    q = ex.get("question", "")
    concept = ex.get("question_concept", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices, clean_labels = [], []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    if isinstance(concept, str) and concept.strip():
        q_text = f"Question: {str(q).strip()}\nConcept: {concept.strip()}"
    else:
        q_text = str(q)

    return q_text, clean_choices, y


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg, split_name):
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{cfg.dataset_name}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(cfg, split_name, hf_split, conceptizer=None):
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(path)

    if conceptizer is None:
        raise RuntimeError("Conceptizer needed because cache is missing.")

    print(f"[cache] building {path}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_commonsenseqa_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(choices.size(0), dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(choices.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path}, rows={len(rows)}, skipped={skipped}")
    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "idx": idx,
        }


def cached_collate(batch):
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_final_hybrid_model(cfg, device):
    if not os.path.exists(cfg.final_hybrid_path):
        raise FileNotFoundError(cfg.final_hybrid_path)

    obj = torch.load(cfg.final_hybrid_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("final_hybrid.pt does not contain key 'model'.")

    model = build_hlcm_from_cfg(cfg).to(device)
    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] final_hybrid.pt loaded from {cfg.final_hybrid_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    if "grpo_best_acc" in obj:
        print(f"[checkpoint] grpo_best_acc = {obj['grpo_best_acc']:.6f}")
        print(f"[checkpoint] grpo_best_acc (%) = {obj['grpo_best_acc'] * 100:.2f}%")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, obj


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(model, x, pad_mask, mu, sigma):
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(model, batch, mu, sigma, cfg):
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu, sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu, sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)
    return logits


# ============================================================
# INFERENCE TIME
# ============================================================

@torch.no_grad()
def run_inference_with_time(model, loader, mu, sigma, cfg, save_path=None):
    model.eval()
    device = next(model.parameters()).device

    total = 0
    correct = 0
    total_loss = 0.0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference CommonsenseQA-{cfg.split}"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)

        labels = batch["label"].to(logits.device, non_blocking=True)
        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)
        total += bs
        correct += int((preds == labels).sum().item())
        total_loss += float(loss.item()) * bs

        for i in range(bs):
            predictions.append({
                "example_index": int(batch["idx"][i].item()),
                "gold": int(labels[i].item()),
                "pred": int(preds[i].item()),
                "correct": int(preds[i].item() == labels[i].item()),
            })

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start

    results = {
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_from_final_hybrid(cfg):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)

    grpo_best_acc, ckpt_obj = read_grpo_best_acc(cfg.final_hybrid_path)

    train_hf, eval_hf, test_hf = load_commonsenseqa(cfg)

    if cfg.split in {"validation", "val", "dev"}:
        hf_split = eval_hf
        split_name = "validation"
    elif cfg.split == "test":
        if test_hf is None:
            raise RuntimeError("No test split available.")
        hf_split = test_hf
        split_name = "test"
    elif cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    else:
        raise ValueError("split must be train, validation, or test")

    cache_path = cache_file_path(cfg, split_name)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=torch.device(cfg.conceptizer_device),
        )

        t0 = time.perf_counter()
        rows = build_or_load_cached_split(cfg, split_name, hf_split, conceptizer)
        cache_build_time_sec = time.perf_counter() - t0

        del conceptizer
        cuda_cleanup()
    else:
        rows = build_or_load_cached_split(cfg, split_name, hf_split, conceptizer=None)

    ds = CachedMCQDataset(rows)
    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_obj = load_final_hybrid_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    out_dir = os.path.join(cfg.out_dir, cfg.dataset_name)
    ensure_dir(out_dir)

    save_path = os.path.join(out_dir, f"final_hybrid_inference_{split_name}_results.json")

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "dataset": cfg.dataset_name,
        "split": split_name,
        "checkpoint": cfg.final_hybrid_path,
        "grpo_best_acc_from_checkpoint": grpo_best_acc,
        "grpo_best_acc_percent_from_checkpoint": None if grpo_best_acc is None else grpo_best_acc * 100,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(out_dir, f"final_hybrid_inference_{split_name}_summary.json")

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [10]:
cfg = InferenceConfig(
    dataset_name="default",

    final_hybrid_path="runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt",
    normalizer_path="normalizer.pt",

    out_dir="runs/hlcm_commonsenseqa_cached_reflogits_fixed",
    cache_dir="commonsenseqa_cached_features",

    split="validation",
    eval_batch_size=2,
    choice_chunk_size=1,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_from_final_hybrid(cfg)

[device] cuda:0
Available keys:
['model', 'stage', 'sft_best_acc', 'grpo_best_acc', 'final_eval_loss', 'final_eval_acc', 'final_test_loss', 'final_test_acc', 'sft_total_minutes', 'grpo_total_minutes', 'max_gpu_alloc_mb', 'dataset', 'arch', 'concept_model', 'chunk_tok_len', 'seq_len', 'finetune_mode', 'n_last_blocks', 'pretrained_ckpt', 'bs_train', 'bs_eval', 'grad_accum_steps', 'use_bf16', 'choice_chunk_size', 'num_train_examples', 'num_eval_examples', 'num_test_examples']

grpo_best_acc = 0.334971
grpo_best_acc (%) = 33.50%


Using the latest cached version of the dataset since tau/commonsense_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/user/.cache/huggingface/datasets/tau___commonsense_qa/default/0.0.0/94630fe30dad47192a8546eb75f094926d47e155 (last modified on Fri Mar 20 18:23:31 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building commonsenseqa_cached_features/default_validation_tok256_seq8.pt


cache:validation: 100%|█████████████████████████████████████████| 1221/1221 [02:44<00:00,  7.44it/s]


[cache] saved commonsenseqa_cached_features/default_validation_tok256_seq8.pt, rows=1221, skipped=0
[load] final_hybrid.pt loaded from runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt
[load] missing keys: 0
[load] unexpected keys: 0
[checkpoint] grpo_best_acc = 0.334971
[checkpoint] grpo_best_acc (%) = 33.50%
[normalizer] loaded


Inference CommonsenseQA-validation: 100%|█████████████████████████| 611/611 [03:08<00:00,  3.23it/s]


[save] results -> runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid_inference_validation_results.json

==================== DONE ====================
{
  "dataset": "default",
  "split": "validation",
  "checkpoint": "runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid.pt",
  "grpo_best_acc_from_checkpoint": 0.33497133497133497,
  "grpo_best_acc_percent_from_checkpoint": 33.4971334971335,
  "num_examples": 1221,
  "correct": 409,
  "loss": 1.5930794556056935,
  "accuracy": 0.33497133497133497,
  "accuracy_percent": 33.4971334971335,
  "cache_build_time_sec": 164.44545110315084,
  "cache_build_time_hms": "00:02:44",
  "inference_time_sec": 189.05423413217068,
  "inference_time_hms": "00:03:09",
  "time_per_example_sec": 0.15483557258982036,
  "examples_per_second": 6.458464184126024
}
[summary saved] runs/hlcm_commonsenseqa_cached_reflogits_fixed/default/final_hybrid_inference_validation_summary.json


In [11]:
print("grpo_best_acc from final_hybrid.pt:", summary["grpo_best_acc_from_checkpoint"])
print("grpo_best_acc %:", summary["grpo_best_acc_percent_from_checkpoint"])
print("Inference accuracy %:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

grpo_best_acc from final_hybrid.pt: 0.33497133497133497
grpo_best_acc %: 33.4971334971335
Inference accuracy %: 33.4971334971335
Inference time: 189.05423413217068
Time/example: 0.15483557258982036
Examples/sec: 6.458464184126024


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class GRPOConfig:
    # --------------------------------------------------------
    # DATASET
    # --------------------------------------------------------
    # Train from auxiliary_train, validate on val, test on test.
    # For cais/mmlu this is the correct pattern when you want actual training data.
    mmlu_dataset_name: str = "cais/mmlu"

    train_config_name: str = "auxiliary_train"
    train_split_candidates: Tuple[str, ...] = ("auxiliary_train", "train")

    eval_config_name: str = "all"
    eval_split_candidates: Tuple[str, ...] = ("validation", "val")

    test_config_name: str = "all"
    test_split_candidates: Tuple[str, ...] = ("test",)

    run_name: str = "aux_train_to_val"

    out_dir: str = "runs/hlcm_mmlu_auxtrain_val_steps"
    cache_dir: str = "mmlu_cached_features"
    ref_logits_dir: str = "mmlu_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 1

    # loader / memory
    train_batch_size: int = 1
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0

    # --------------------------------------------------------
    # STEP-BASED TRAINING
    # --------------------------------------------------------
    sft_max_steps: int = 300
    sft_eval_every: int = 100
    sft_save_every: int = 100
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    grpo_max_steps: int = 150
    grpo_eval_every: int = 100
    grpo_save_every: int = 100
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # policy / reward
    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # choice chunking
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "head_only":
        raise ValueError(
            "finetune_mode='head_only' is invalid for this script because no separate trainable head exists. "
            "Use 'last_blocks' or 'full'."
        )
    elif mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# MMLU RAW / NORMALIZATION
# ============================================================

LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3}


def answer_to_index(answer_value: Any, num_choices: int) -> int:
    if isinstance(answer_value, int):
        return max(0, min(int(answer_value), num_choices - 1))

    if isinstance(answer_value, str):
        key = answer_value.strip().upper()
        if key in LETTER_TO_INDEX:
            return max(0, min(LETTER_TO_INDEX[key], num_choices - 1))

    return 0


def _try_load_split(dataset_name: str, config_name: str, split_candidates: Tuple[str, ...]):
    last_err = None
    for split_name in split_candidates:
        try:
            ds = load_dataset(dataset_name, config_name, split=split_name)
            print(f"[data] loaded {dataset_name} config={config_name} split={split_name} n={len(ds)}")
            return ds, split_name
        except Exception as e:
            last_err = e
    raise ValueError(
        f"Could not load any split from candidates={split_candidates} for "
        f"dataset={dataset_name}, config={config_name}. Last error: {last_err}"
    )


def load_mmlu_train_eval_test(cfg: GRPOConfig):
    train_split, train_split_name = _try_load_split(
        cfg.mmlu_dataset_name,
        cfg.train_config_name,
        cfg.train_split_candidates,
    )

    eval_split, eval_split_name = _try_load_split(
        cfg.mmlu_dataset_name,
        cfg.eval_config_name,
        cfg.eval_split_candidates,
    )

    test_split = None
    test_split_name = None
    try:
        test_split, test_split_name = _try_load_split(
            cfg.mmlu_dataset_name,
            cfg.test_config_name,
            cfg.test_split_candidates,
        )
    except Exception as e:
        print(f"[data] test split not loaded: {e}")

    return train_split, eval_split, test_split, train_split_name, eval_split_name, test_split_name


def normalize_mmlu_example(ex: Dict[str, Any], min_valid_choices: int):
    """
    Supports examples like:
    {
      "question": "...",
      "choices": ["...", "...", "...", "..."],
      "answer": "D"
    }

    and also integer answers.
    """
    q = str(ex.get("question", "")).strip()
    raw_choices = ex.get("choices", [])
    answer = ex.get("answer", "A")

    clean_choices = []
    if isinstance(raw_choices, list):
        for c in raw_choices:
            if isinstance(c, str) and c.strip():
                clean_choices.append(c.strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]

    y = answer_to_index(answer, len(clean_choices))
    y = max(0, min(y, len(clean_choices) - 1))

    q_text = f"Question: {q}"
    return q_text, clean_choices, y


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(
    cfg: GRPOConfig,
    cache_tag: str,
) -> str:
    safe_tag = cache_tag.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, split_tag: str) -> str:
    safe_tag = split_tag.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_tag}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    cache_tag: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, cache_tag)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {cache_tag}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{cache_tag}"):
        try:
            q_text, choice_texts, label = normalize_mmlu_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for j, ct in enumerate(choice_texts):
                letter = chr(ord("A") + j)
                qc = f"{q_text}\nAnswer Choice {letter}: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {cache_tag}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, choice_chunk_size)
    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
        choice_chunk_size=cfg.choice_chunk_size,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


def infinite_loader(loader: DataLoader):
    while True:
        for batch in loader:
            yield batch


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT (STEP-BASED)
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    total_steps = max(1, cfg.sft_max_steps)
    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "step": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()

    data_iter = infinite_loader(train_loader)
    model.train()
    opt.zero_grad(set_to_none=True)

    running_loss_sum = 0.0
    running_acc_sum = 0.0
    running_count = 0

    progress = tqdm(range(1, total_steps + 1), desc="sft", dynamic_ncols=True)

    for global_opt_step in progress:
        micro_loss_sum = 0.0
        micro_acc_sum = 0.0
        micro_count = 0

        for _ in range(cfg.grad_accum_steps):
            batch = next(data_iter)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            micro_loss_sum += float(loss.item()) * bs
            micro_acc_sum += float(acc.item()) * bs
            micro_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

        if scaler.is_enabled():
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            scaler.step(opt)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            opt.step()

        sched.step()
        opt.zero_grad(set_to_none=True)

        running_loss_sum += micro_loss_sum
        running_acc_sum += micro_acc_sum
        running_count += micro_count

        train_loss = running_loss_sum / max(1, running_count)
        train_acc = running_acc_sum / max(1, running_count)

        progress.set_postfix(
            loss=f"{train_loss:.4f}",
            acc=f"{train_acc:.4f}",
            lr=f"{opt.param_groups[0]['lr']:.2e}",
        )

        append_dict_to_csv(train_csv, {
            "step": global_opt_step,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
        })

        if (global_opt_step % cfg.sft_eval_every == 0) or (global_opt_step == total_steps):
            ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
            cuda_cleanup()

            append_dict_to_csv(eval_csv, {
                "phase": "eval",
                "step": global_opt_step,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "eval_loss": ev["loss"],
                "eval_acc": ev["acc"],
            })

            print(
                f"[SFT][step {global_opt_step}/{total_steps}] "
                f"train_loss={train_loss:.4f} "
                f"train_acc={train_acc:.4f} "
                f"eval_loss={ev['loss']:.4f} "
                f"eval_acc={ev['acc']:.4f}"
            )

            if ev["acc"] > best_acc:
                best_acc = ev["acc"]
                best_state = clone_state_dict_to_cpu(model)
                save_checkpoint(
                    {
                        "model": best_state,
                        "stage": "sft_best",
                        "best_eval_acc": best_acc,
                        "global_opt_step": global_opt_step,
                        **metadata,
                    },
                    os.path.join(out_dir, "sft_best.pt"),
                )
                print("  saved sft_best.pt")

        if (global_opt_step % cfg.sft_save_every == 0) or (global_opt_step == total_steps):
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "sft_last",
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_last.pt"),
            )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS (STEP-BASED)
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    total_steps = max(1, cfg.grpo_max_steps)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "step": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()

    data_iter = infinite_loader(train_loader)
    model.train()
    opt.zero_grad(set_to_none=True)

    running_loss_sum = 0.0
    running_acc_sum = 0.0
    running_count = 0
    last_stats = None

    progress = tqdm(range(1, total_steps + 1), desc="grpo", dynamic_ncols=True)

    for global_opt_step in progress:
        micro_loss_sum = 0.0
        micro_acc_sum = 0.0
        micro_count = 0

        for _ in range(cfg.grad_accum_steps):
            batch = next(data_iter)

            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            micro_loss_sum += float(stats["loss"]) * bs
            micro_acc_sum += float(stats["acc"]) * bs
            micro_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

        if scaler.is_enabled():
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            scaler.step(opt)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            opt.step()

        sched.step()
        opt.zero_grad(set_to_none=True)

        running_loss_sum += micro_loss_sum
        running_acc_sum += micro_acc_sum
        running_count += micro_count

        train_loss = running_loss_sum / max(1, running_count)
        train_acc = running_acc_sum / max(1, running_count)

        progress.set_postfix(
            loss=f"{train_loss:.4f}",
            acc=f"{train_acc:.4f}",
            policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
            kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
            lr=f"{opt.param_groups[0]['lr']:.2e}",
        )

        append_dict_to_csv(train_csv, {
            "step": global_opt_step,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
        })

        if (global_opt_step % cfg.grpo_eval_every == 0) or (global_opt_step == total_steps):
            ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
            cuda_cleanup()

            append_dict_to_csv(eval_csv, {
                "phase": "eval",
                "step": global_opt_step,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "eval_loss": ev["loss"],
                "eval_acc": ev["acc"],
            })

            print(
                f"[GRPO][step {global_opt_step}/{total_steps}] "
                f"train_loss={train_loss:.4f} "
                f"train_acc={train_acc:.4f} "
                f"eval_loss={ev['loss']:.4f} "
                f"eval_acc={ev['acc']:.4f}"
            )

            if ev["acc"] > best_acc:
                best_acc = ev["acc"]
                best_state = clone_state_dict_to_cpu(model)
                save_checkpoint(
                    {
                        "model": best_state,
                        "stage": "grpo_best",
                        "best_eval_acc": best_acc,
                        "global_opt_step": global_opt_step,
                        **metadata,
                    },
                    os.path.join(out_dir, "grpo_best.pt"),
                )
                print("  saved grpo_best.pt")

        if (global_opt_step % cfg.grpo_save_every == 0) or (global_opt_step == total_steps):
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "grpo_last",
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_last.pt"),
            )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN
# ============================================================

def train_mmlu_hybrid_hlcm(cfg: GRPOConfig, device: torch.device):
    print(f"\n==================== {cfg.run_name} ====================")
    out_dir = os.path.join(cfg.out_dir, cfg.run_name)
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf, train_split_name, eval_split_name, test_split_name = load_mmlu_train_eval_test(cfg)

    train_cache_tag = f"{cfg.mmlu_dataset_name}_{cfg.train_config_name}_{train_split_name}"
    eval_cache_tag = f"{cfg.mmlu_dataset_name}_{cfg.eval_config_name}_{eval_split_name}"
    test_cache_tag = (
        f"{cfg.mmlu_dataset_name}_{cfg.test_config_name}_{test_split_name}"
        if test_hf is not None and test_split_name is not None else None
    )

    train_rows = build_or_load_cached_split(cfg, train_cache_tag, train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, eval_cache_tag, eval_hf, conceptizer)
    test_rows = build_or_load_cached_split(cfg, test_cache_tag, test_hf, conceptizer) if test_hf is not None else []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metadata_base = {
        "run_name": cfg.run_name,
        "train_source": f"{cfg.mmlu_dataset_name}:{cfg.train_config_name}:{train_split_name}",
        "eval_source": f"{cfg.mmlu_dataset_name}:{cfg.eval_config_name}:{eval_split_name}",
        "test_source": None if test_split_name is None else f"{cfg.mmlu_dataset_name}:{cfg.test_config_name}:{test_split_name}",
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "num_test_examples": len(test_rows),
    }

    print(f"\n========== {cfg.run_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_max_steps": cfg.sft_max_steps,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, train_cache_tag)
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    print(f"\n========== {cfg.run_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_max_steps": cfg.grpo_max_steps,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    final_test = evaluate_hlcm(hlcm, test_loader, mu, sigma, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "final_test_loss": None if final_test is None else final_test["loss"],
            "final_test_acc": None if final_test is None else final_test["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "run_name": cfg.run_name,
        "train_source": metadata_base["train_source"],
        "eval_source": metadata_base["eval_source"],
        "test_source": metadata_base["test_source"],
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "sft_max_steps": cfg.sft_max_steps,
        "grpo_max_steps": cfg.grpo_max_steps,
        "ref_logits_path": ref_logits_path,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] run={cfg.run_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = GRPOConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(f"Train source: {cfg.mmlu_dataset_name}:{cfg.train_config_name}:{cfg.train_split_candidates}")
    print(f"Eval source: {cfg.mmlu_dataset_name}:{cfg.eval_config_name}:{cfg.eval_split_candidates}")
    print(
        f"SFT max_steps={cfg.sft_max_steps}, GRPO max_steps={cfg.grpo_max_steps}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()
    train_mmlu_hybrid_hlcm(cfg, device)
    total_all = (time.time() - all_t0) / 60.0

    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
Train source: cais/mmlu:auxiliary_train:('auxiliary_train', 'train')
Eval source: cais/mmlu:all:('validation', 'val')
SFT max_steps=300, GRPO max_steps=150, bs_train=1, bs_eval=2, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
choice_chunk_size=1

==================== aux_train_to_val ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[data] loaded cais/mmlu config=auxiliary_train split=train n=99842


Using the latest cached version of the dataset since cais/mmlu couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /home/user/.cache/huggingface/datasets/cais___mmlu/all/0.0.0/c30699e8356da336a370243923dbaf21066bb9fe (last modified on Sun Mar 22 17:27:07 2026).


...

[GRPO][step 100/150] train_loss=-0.0760 train_acc=0.8263 eval_loss=1.3902 eval_acc=0.2489


grpo:  99%|▉| 149/150 [15:12<00:04,  4.30s/it, acc=0.8842, kl=0.6268, loss=-0.0835, lr=0.00e+00, pol

[GRPO][step 150/150] train_loss=-0.0835 train_acc=0.8842 eval_loss=1.3907 eval_acc=0.2476


grpo: 100%|█| 150/150 [18:50<00:00,  7.54s/it, acc=0.8842, kl=0.6268, loss=-0.0835, lr=0.00e+00, pol


[GRPO][FINAL] best_acc=0.2528 total_train_time=18.85 min
[HYBRID][FINAL] run=aux_train_to_val sft_best_acc=0.2528 grpo_best_acc=0.2528 final_eval_acc=0.2528 final_test_acc=0.2460 max_gpu_alloc=25474.9 MB saved -> runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val

All done. Outputs in: runs/hlcm_mmlu_auxtrain_val_steps
Total wall time: 361.04 min


In [1]:

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    # MMLU eval/test config
    mmlu_dataset_name: str = "cais/mmlu"

    eval_config_name: str = "all"
    eval_split_candidates: Tuple[str, ...] = ("validation", "val")

    test_config_name: str = "all"
    test_split_candidates: Tuple[str, ...] = ("test",)

    run_name: str = "aux_train_to_val"

    out_dir: str = "runs/hlcm_mmlu_auxtrain_val_steps"
    cache_dir: str = "mmlu_cached_features"

    # checkpoint to evaluate
    eval_ckpt_path: str = "runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt"

    # base pretrained HLCM, needed only to construct architecture before loading grpo_last.pt
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM architecture
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # eval
    eval_batch_size: int = 2
    num_workers: int = 0

    # scoring
    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; continuing without normalizer")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# MMLU LOADING / NORMALIZATION
# ============================================================

LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}


def answer_to_index(answer_value: Any, num_choices: int) -> int:
    if isinstance(answer_value, int):
        return max(0, min(int(answer_value), num_choices - 1))

    if isinstance(answer_value, str):
        key = answer_value.strip().upper()

        if key in LETTER_TO_INDEX:
            return max(0, min(LETTER_TO_INDEX[key], num_choices - 1))

        if key.isdigit():
            idx = int(key)
            return max(0, min(idx, num_choices - 1))

    return 0


def try_load_split(dataset_name: str, config_name: str, split_candidates: Tuple[str, ...]):
    last_err = None

    for split_name in split_candidates:
        try:
            ds = load_dataset(dataset_name, config_name, split=split_name)
            print(f"[data] loaded {dataset_name} config={config_name} split={split_name} n={len(ds)}")
            return ds, split_name
        except Exception as e:
            last_err = e

    raise ValueError(
        f"Could not load split candidates={split_candidates} "
        f"for dataset={dataset_name}, config={config_name}. Last error: {last_err}"
    )


def load_mmlu_eval_test(cfg: EvalConfig):
    eval_split, eval_split_name = try_load_split(
        cfg.mmlu_dataset_name,
        cfg.eval_config_name,
        cfg.eval_split_candidates,
    )

    test_split = None
    test_split_name = None

    try:
        test_split, test_split_name = try_load_split(
            cfg.mmlu_dataset_name,
            cfg.test_config_name,
            cfg.test_split_candidates,
        )
    except Exception as e:
        print(f"[data] test split not loaded: {e}")

    return eval_split, test_split, eval_split_name, test_split_name


def normalize_mmlu_example(ex: Dict[str, Any], min_valid_choices: int):
    q = str(ex.get("question", "")).strip()
    raw_choices = ex.get("choices", [])
    answer = ex.get("answer", "A")

    clean_choices = []

    if isinstance(raw_choices, list):
        for c in raw_choices:
            if isinstance(c, str) and c.strip():
                clean_choices.append(c.strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]

    y = answer_to_index(answer, len(clean_choices))
    y = max(0, min(y, len(clean_choices) - 1))

    q_text = f"Question: {q}"
    return q_text, clean_choices, y


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: EvalConfig, cache_tag: str) -> str:
    safe_tag = cache_tag.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: EvalConfig,
    cache_tag: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, cache_tag)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {cache_tag}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{cache_tag}"):
        try:
            q_text, choice_texts, label = normalize_mmlu_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []

            for j, ct in enumerate(choice_texts):
                letter = chr(ord("A") + j)
                qc = f"{q_text}\nAnswer Choice {letter}: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {cache_tag}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Base checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] base HLCM loaded from {cfg.ckpt_path}")
    print(f"[load] base missing keys: {len(missing)}")
    print(f"[load] base unexpected keys: {len(unexpected)}")

    return model


def load_eval_checkpoint(model: HyperbolicLCM, ckpt_path: str, device: torch.device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Eval checkpoint not found: {ckpt_path}")

    obj = torch.load(ckpt_path, map_location="cpu")

    if isinstance(obj, dict) and "model" in obj:
        state = obj["model"]
    elif isinstance(obj, dict) and "model_state" in obj:
        state = obj["model_state"]
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] eval checkpoint loaded from {ckpt_path}")
    print(f"[load] eval missing keys: {len(missing)}")
    print(f"[load] eval unexpected keys: {len(unexpected)}")

    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    info = {}

    if isinstance(obj, dict):
        for key in [
            "stage",
            "best_eval_acc",
            "global_opt_step",
            "run_name",
            "train_source",
            "eval_source",
            "test_source",
        ]:
            if key in obj:
                info[key] = obj[key]

    return info


# ============================================================
# LOGITS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
    choice_chunk_size: int = 1,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, int(choice_chunk_size))

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        ec = ec.reshape(B, (k1 - k0), -1)

        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits


# ============================================================
# RANKING METRICS
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4, 5),
    ece_bins: int = 15,
) -> Dict[str, float]:
    if loader is None:
        return {}

    model.eval()

    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0
    total_nll = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}

    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
            choice_chunk_size=cfg.choice_chunk_size,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        batch_size = labels.size(0)
        total += batch_size
        total_loss += float(loss.item()) * batch_size
        total_nll += float(loss.item()) * batch_size

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        pred_conf = probs.max(dim=-1).values.detach().cpu()
        all_confidences.append(pred_conf)
        all_correctness.append(batch_correct.detach().cpu().float())

        brier_per_example = multiclass_brier_score(
            probs=probs,
            labels=labels,
            choice_mask=choice_mask,
        )
        total_brier += float(brier_per_example.sum().item())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0

                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences = torch.cat(all_confidences, dim=0) if all_confidences else torch.empty(0)
    correctness = torch.cat(all_correctness, dim=0) if all_correctness else torch.empty(0)

    ece, mce = expected_calibration_error(
        confidences=confidences,
        correctness=correctness,
        n_bins=ece_bins,
    )

    results = {
        "loss": total_loss / max(total, 1),
        "nll": total_nll / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": ece_bins,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results

def multiclass_brier_score(probs: torch.Tensor, labels: torch.Tensor, choice_mask: torch.Tensor) -> torch.Tensor:
    """
    Multiclass Brier score:
        sum_k (p_k - y_k)^2

    probs:       [B, K]
    labels:      [B]
    choice_mask: [B, K]
    """
    B, K = probs.shape
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)

    sq_error = (probs - one_hot) ** 2
    sq_error = sq_error.masked_fill(~choice_mask, 0.0)

    return sq_error.sum(dim=1)


def expected_calibration_error(
    confidences: torch.Tensor,
    correctness: torch.Tensor,
    n_bins: int = 15,
) -> Tuple[float, float]:
    """
    ECE and MCE from prediction confidence.

    confidences: [N]
    correctness: [N], values 0 or 1
    """
    ece = 0.0
    mce = 0.0
    n = max(1, confidences.numel())

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            weight = mask.float().mean().item()

            ece += weight * gap
            mce = max(mce, gap)

    return float(ece), float(mce)

# ============================================================
# MAIN EVAL
# ============================================================

def main():
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("This script is EVAL ONLY. It does not train.")
    print("Checkpoint:", cfg.eval_ckpt_path)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    eval_hf, test_hf, eval_split_name, test_split_name = load_mmlu_eval_test(cfg)

    eval_cache_tag = f"{cfg.mmlu_dataset_name}_{cfg.eval_config_name}_{eval_split_name}"
    test_cache_tag = (
        f"{cfg.mmlu_dataset_name}_{cfg.test_config_name}_{test_split_name}"
        if test_hf is not None and test_split_name is not None else None
    )

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        cache_tag=eval_cache_tag,
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    test_rows = []
    if test_hf is not None:
        test_rows = build_or_load_cached_split(
            cfg=cfg,
            cache_tag=test_cache_tag,
            hf_split=test_hf,
            conceptizer=conceptizer,
        )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_loader = DataLoader(
            CachedMCQDataset(test_rows),
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    model = load_base_hlcm(cfg, device)
    ckpt_info = load_eval_checkpoint(model, cfg.eval_ckpt_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    validation_metrics = evaluate_ranking_metrics_hlcm(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4, 5),
    )

    test_metrics = None
    if test_loader is not None:
        test_metrics = evaluate_ranking_metrics_hlcm(
            model=model,
            loader=test_loader,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            k_values=(1, 2, 3, 4, 5),
        )

    summary = {
        "run_name": cfg.run_name,
        "checkpoint": cfg.eval_ckpt_path,
        "checkpoint_info": ckpt_info,
        "eval_source": f"{cfg.mmlu_dataset_name}:{cfg.eval_config_name}:{eval_split_name}",
        "test_source": None if test_split_name is None else f"{cfg.mmlu_dataset_name}:{cfg.test_config_name}:{test_split_name}",
        "num_validation_examples": len(eval_rows),
        "num_test_examples": len(test_rows),
        "validation": validation_metrics,
        "test": test_metrics,
    }

    save_dir = os.path.join(cfg.out_dir, cfg.run_name)
    ensure_dir(save_dir)

    save_path = os.path.join(save_dir, "grpo_last_eval_precision_recall_ranking_brier_ece.json")

    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    del model
    cuda_cleanup()


if __name__ == "__main__":
    main()

Device: cuda:0
This script is EVAL ONLY. It does not train.
Checkpoint: runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt


Using the latest cached version of the dataset since cais/mmlu couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /home/user/.cache/huggingface/datasets/cais___mmlu/all/0.0.0/c30699e8356da336a370243923dbaf21066bb9fe (last modified on Mon Apr 27 14:18:37 2026).


[data] loaded cais/mmlu config=all split=validation n=1531


Using the latest cached version of the dataset since cais/mmlu couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /home/user/.cache/huggingface/datasets/cais___mmlu/all/0.0.0/c30699e8356da336a370243923dbaf21066bb9fe (last modified on Mon Apr 27 14:18:37 2026).


[data] loaded cais/mmlu config=all split=test n=14042


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading mmlu_cached_features/cais_mmlu_all_validation_tok256_seq8.pt
[cache] loading mmlu_cached_features/cais_mmlu_all_test_tok256_seq8.pt
[load] base HLCM loaded from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] base missing keys: 0
[load] base unexpected keys: 0
[load] eval checkpoint loaded from runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt
[load] eval missing keys: 0
[load] eval unexpected keys: 0
[normalizer] loaded normalizer.pt


{
  "run_name": "aux_train_to_val",
  "checkpoint": "runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt",
  "checkpoint_info": {
    "stage": "grpo_last",
    "global_opt_step": 150,
    "run_name": "aux_train_to_val",
    "train_source": "cais/mmlu:auxiliary_train:train",
    "eval_source": "cais/mmlu:all:validation",
    "test_source": "cais/mmlu:all:test"
  },
  "eval_source": "cais/mmlu:all:validation",
  "test_source": "cais/mmlu:all:test",
  "num_validation_examples": 1531,
  "num_test_examples": 14042,
  "validation": {
    "loss": 1.3906727278645252,
    "nll": 1.3906727278645252,
    "accuracy": 0.24755062050947094,
    "brier_score": 0.7518841697451805,
    "ece": 0.02843408012137219,
    "mce": 0.24767056107521057,
    "ece_bins": 15,
    "precision@1": 0.24755062050947094,
    "recall@1": 0.24755062050947094,
    "precision@2": 0.2550620509470934,
    "recall@2": 0.5101241018941868,
    "precision@3": 0.25016329196603243,
    "recall@3": 0.7504898758981058,
   

In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    mmlu_dataset_name: str = "cais/mmlu"

    eval_config_name: str = "all"
    eval_split_candidates: Tuple[str, ...] = ("validation", "val")

    test_config_name: str = "all"
    test_split_candidates: Tuple[str, ...] = ("test",)

    run_name: str = "aux_train_to_val"

    out_dir: str = "runs/hlcm_mmlu_auxtrain_val_steps"
    cache_dir: str = "mmlu_cached_features"

    grpo_ckpt_path: str = "runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 2
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1
    choice_chunk_size: int = 1

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0

    split: str = "test"   # "validation" or "test"
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break

        return chunks[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# MMLU DATA
# ============================================================

LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3}


def answer_to_index(answer_value: Any, num_choices: int) -> int:
    if isinstance(answer_value, int):
        return max(0, min(int(answer_value), num_choices - 1))

    if isinstance(answer_value, str):
        key = answer_value.strip().upper()
        if key in LETTER_TO_INDEX:
            return max(0, min(LETTER_TO_INDEX[key], num_choices - 1))

    return 0


def try_load_split(dataset_name: str, config_name: str, split_candidates: Tuple[str, ...]):
    last_err = None

    for split_name in split_candidates:
        try:
            ds = load_dataset(dataset_name, config_name, split=split_name)
            print(f"[data] loaded {dataset_name} config={config_name} split={split_name} n={len(ds)}")
            return ds, split_name
        except Exception as e:
            last_err = e

    raise ValueError(
        f"Could not load split from {split_candidates} for "
        f"{dataset_name}:{config_name}. Last error: {last_err}"
    )


def load_mmlu_eval_or_test(cfg: InferenceConfig):
    if cfg.split in {"validation", "val", "dev"}:
        hf_split, split_name = try_load_split(
            cfg.mmlu_dataset_name,
            cfg.eval_config_name,
            cfg.eval_split_candidates,
        )
        return hf_split, split_name

    if cfg.split == "test":
        hf_split, split_name = try_load_split(
            cfg.mmlu_dataset_name,
            cfg.test_config_name,
            cfg.test_split_candidates,
        )
        return hf_split, split_name

    raise ValueError("cfg.split must be either 'validation' or 'test'")


def normalize_mmlu_example(ex: Dict[str, Any], min_valid_choices: int):
    q = str(ex.get("question", "")).strip()
    raw_choices = ex.get("choices", [])
    answer = ex.get("answer", "A")

    clean_choices = []

    if isinstance(raw_choices, list):
        for c in raw_choices:
            if isinstance(c, str) and c.strip():
                clean_choices.append(c.strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]

    y = answer_to_index(answer, len(clean_choices))
    y = max(0, min(y, len(clean_choices) - 1))

    q_text = f"Question: {q}"

    return q_text, clean_choices, y


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: InferenceConfig, cache_tag: str) -> str:
    safe_tag = cache_tag.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    cache_tag: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
):
    path = cache_file_path(cfg, cache_tag)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(path)

    if conceptizer is None:
        raise RuntimeError("Conceptizer is required because cache is missing.")

    print(f"[cache] building {cache_tag}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{cache_tag}"):
        try:
            q_text, choice_texts, label = normalize_mmlu_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for j, ct in enumerate(choice_texts):
                letter = chr(ord("A") + j)
                qc = f"{q_text}\nAnswer Choice {letter}: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(choices.size(0), dtype=torch.bool)

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": choices,
                    "cmask": cmask,
                    "choice_mask": choice_mask,
                    "label": int(label),
                    "num_choices": int(choices.size(0)),
                }
            )

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {cache_tag}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch):
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]

        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_last_model(cfg: InferenceConfig, device: torch.device):
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(cfg.grpo_ckpt_path)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_last.pt does not contain key 'model'.")

    model = build_hlcm_from_cfg(cfg).to(device)

    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded model from {cfg.grpo_ckpt_path}")
    print(f"[load] stage: {obj.get('stage', 'unknown')}")
    print(f"[load] global_opt_step: {obj.get('global_opt_step', 'unknown')}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model, obj


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(
    model,
    x,
    pad_mask,
    mu,
    sigma,
):
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(
    model,
    batch,
    mu,
    sigma,
    cfg,
):
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu, sigma)
    e_q = F.normalize(e_q, dim=-1)

    e_c_chunks = []
    chunk_sz = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, chunk_sz):
        k1 = min(K, k0 + chunk_sz)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu, sigma)
        ec = ec.reshape(B, (k1 - k0), -1)
        e_c_chunks.append(ec)

    e_c = torch.cat(e_c_chunks, dim=1)
    e_c = F.normalize(e_c, dim=-1)

    logits = torch.einsum("bd,bkd->bk", e_q, e_c)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)

    return logits


# ============================================================
# INFERENCE + TIMING
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model,
    loader,
    mu,
    sigma,
    cfg,
    save_path=None,
):
    model.eval()
    device = next(model.parameters()).device

    total = 0
    correct = 0
    total_loss = 0.0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference MMLU-{cfg.split}"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)

        total += int(bs)
        correct += int((preds == labels).sum().item())
        total_loss += float(loss.item()) * int(bs)

        for i in range(bs):
            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(preds[i].item() == labels[i].item()),
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_mmlu(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[checkpoint]", cfg.grpo_ckpt_path)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    hf_split, split_name = load_mmlu_eval_or_test(cfg)

    cache_tag = f"{cfg.mmlu_dataset_name}_{cfg.eval_config_name if split_name != 'test' else cfg.test_config_name}_{split_name}"

    cache_path = cache_file_path(cfg, cache_tag)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=torch.device(cfg.conceptizer_device),
        )

        t_cache = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            cache_tag=cache_tag,
            hf_split=hf_split,
            conceptizer=conceptizer,
        )

        cache_build_time_sec = time.perf_counter() - t_cache

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            cache_tag=cache_tag,
            hf_split=hf_split,
            conceptizer=None,
        )

    print(f"[data] split={split_name}, examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_obj = load_grpo_last_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    out_dir = os.path.join(cfg.out_dir, cfg.run_name)
    ensure_dir(out_dir)

    save_path = os.path.join(out_dir, f"grpo_last_inference_{split_name}_results.json")

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "run_name": cfg.run_name,
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "checkpoint_stage": ckpt_obj.get("stage", None),
        "checkpoint_global_opt_step": ckpt_obj.get("global_opt_step", None),
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(out_dir, f"grpo_last_inference_{split_name}_summary.json")

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== GRPO LAST INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [2]:
cfg = InferenceConfig(
    grpo_ckpt_path="runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt",
    normalizer_path="normalizer.pt",

    out_dir="runs/hlcm_mmlu_auxtrain_val_steps",
    cache_dir="mmlu_cached_features",

    split="test",          # change to "validation" if needed
    eval_batch_size=2,
    choice_chunk_size=1,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_mmlu(cfg)

[device] cuda:0
[checkpoint] runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt


Using the latest cached version of the dataset since cais/mmlu couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /home/user/.cache/huggingface/datasets/cais___mmlu/all/0.0.0/c30699e8356da336a370243923dbaf21066bb9fe (last modified on Mon Apr 27 14:18:37 2026).


[data] loaded cais/mmlu config=all split=test n=14042
[cache] not found: mmlu_cached_features/cais_mmlu_all_test_tok256_seq8.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building cais/mmlu_all_test


cache:cais/mmlu_all_test: 100%|███████████████████████████████| 14042/14042 [34:52<00:00,  6.71it/s]


[cache] saved mmlu_cached_features/cais_mmlu_all_test_tok256_seq8.pt (14042 examples, skipped=0)
[data] split=test, examples=14042
[cache] mmlu_cached_features/cais_mmlu_all_test_tok256_seq8.pt
[load] loaded model from runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt
[load] stage: grpo_last
[load] global_opt_step: 150
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference MMLU-test: 100%|██████████████████████████████████████| 7021/7021 [30:28<00:00,  3.84it/s]


[save] results -> runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last_inference_test_results.json

==================== GRPO LAST INFERENCE DONE ====================
{
  "run_name": "aux_train_to_val",
  "split": "test",
  "checkpoint": "runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last.pt",
  "checkpoint_stage": "grpo_last",
  "checkpoint_global_opt_step": 150,
  "cache_file": "mmlu_cached_features/cais_mmlu_all_test_tok256_seq8.pt",
  "num_examples": 14042,
  "correct": 3447,
  "loss": 1.3924945774809823,
  "accuracy": 0.2454778521578123,
  "accuracy_percent": 24.54778521578123,
  "cache_build_time_sec": 2095.562712376006,
  "cache_build_time_hms": "00:34:55",
  "inference_time_sec": 1828.584918397013,
  "inference_time_hms": "00:30:28",
  "time_per_example_sec": 0.13022254083442622,
  "examples_per_second": 7.67916209891395
}
[summary saved] runs/hlcm_mmlu_auxtrain_val_steps/aux_train_to_val/grpo_last_inference_test_summary.json


In [3]:
print("Accuracy:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

Accuracy: 24.54778521578123
Inference time: 1828.584918397013
Time/example: 0.13022254083442622
Examples/sec: 7.67916209891395
